# Boolean Functions to Quantum Oracles: Structure and Conditional Workspace

This notebook follows a Boolean function from a public circuit description to a reversible quantum computation. We compare **XAG**, **AIG**, **Bennett**, **k-LUT LHRS**, and **best-fit LHRS**, then examine where a qubit might serve as conditional workspace.

Start with three tiny functions whose complete circuits fit on a page. Then run the same flow on the original EPFL `ctrl`, `int2float`, and `cavlc` benchmarks. The last experiment borrows an input qubit during a four-control operation and checks that its state and phase are restored.

The circuits are local simulations and synthesis results; no IBM Quantum jobs are submitted.

## 1. Environment and the two library versions

Use the repository's Python 3.10 environment. From the repository root, install the notebook dependencies if needed:

```bash
source .venv/bin/activate
python -m pip install -r pyproject.toml
python -m pip install -e .
python -m ipykernel install --user --name logique --display-name "Logique Python 3.10"
```

Select that kernel. The repository's tweedledum Python extension must already be installed following its own build instructions. CMake and a C++17 compiler are needed for the native bridge.

Caterpillar uses an older tweedledum API than this repository's installed extension. The helper builds against its own pinned dependencies and exchanges JSON with Python. This keeps the two versions separate. The first build downloads reference sources; subsequent runs use the local build.

The bridge applies two documented reference-library fixes in a local header overlay: uncomputation must remove released wire mappings, and eager cleanup must use mapped-cell fanout counts rather than original-gate counts. Otherwise nested best-fit computation can reuse stale mappings or erase a value too early. The optional router regression exercises these cases; fixes and the reference revision are recorded below.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Math, SVG, HTML
from logique.synthesis import synthesize, METHODS
from logique.paths import native_executable, cache_directory
from logique.benchmarks.datasets import download_benchmarks, benchmark_manifest
from logique.benchmarks.provenance import environment
from logique.benchmarks.export import export_result
from logique.benchmarks.runner import new_run_directory
from logique.verification import validate, truth_outputs, verify_small_quantum
from logique.verification.classical import input_assignments
from logique.workspace import workspace_analysis, borrowing_demo, verify_borrowing
from logique.visualization.circuits import network_figure, circuit_window
from logique.visualization import boolean as bv
ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
OUT = Path(os.environ["LOGIQUE_OUTPUT"]) if "LOGIQUE_OUTPUT" in os.environ else new_run_directory()
OFFLINE = os.environ.get("LOGIQUE_OFFLINE", "0") == "1"
native_executable()  # Build explicitly during setup; notebooks never compile dependencies.
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

display(pd.Series(environment(), name="version"))


## 2. Download a small public benchmark set

The [EPFL Combinational Benchmark Suite](https://github.com/lsils/benchmarks) supplies the same combinational designs in several formats. We use AIGER for machine import and retain Verilog for reading. The download manifest pins a Git revision and SHA-256 hashes, including the MIT license.

| Benchmark | Inputs | Outputs | Why start here? |
|---|---:|---:|---|
| `ctrl` | 7 | 26 | Small input space; many related outputs |
| `int2float` | 11 | 7 | Conversion logic with several output cones |
| `cavlc` | 10 | 11 | A larger control/coding example |

“Small” here refers to the Boolean input space, not the number of quantum wires. Synthesizing intermediate values can require hundreds of wires. We enumerate all 128, 2048, or 1024 classical inputs for correctness checks, but never allocate statevectors for those large quantum circuits.

In [ ]:
benchmarks = download_benchmarks(offline=OFFLINE)
manifest = benchmark_manifest()
print("EPFL revision:", manifest["revision"])
display(pd.DataFrame([
    {"benchmark": name, "AIGER bytes": path.stat().st_size,
     "Verilog": str(path.with_suffix(".v"))}
    for name, path in benchmarks.items()
]))
print("Cache:", cache_directory())

## 3. Representation and scheduling are different decisions

**AIG** means AND-inverter graph. Each internal node performs a two-input AND; an edge can invert its signal. An OR can be represented using De Morgan's rule. An XOR needs several AND/inverter operations.

**XAG** adds XOR nodes. This matters because a parity can be assembled with CNOT gates, while an AND needs a nonlinear controlled operation. Specialized XAG synthesis can temporarily modify a parity wire and restore it afterward.

**Bennett** is a compute/uncompute schedule, not a graph format. Compute intermediate nodes, keep the required outputs, and undo temporary nodes in reverse order. We apply it to both AIGs and XAGs.

**k-LUT** is another graph format: a node stores any Boolean function of at most k inputs. **LHRS**, LUT-based hierarchical reversible synthesis, synthesizes these small functions individually and schedules their cleanup. A larger k can reduce the number of stored intermediate values, but each LUT may expand into more gates or larger controls.

**Best-fit LHRS** first forms larger cells and uses an eager cleanup schedule. Inside each cell it chooses a smaller LUT size when the currently available clean workspace can hold the additional intermediates. It is the reference Caterpillar strategy, not a new “best” option chosen by our wrapper. See [Caterpillar's strategy documentation](https://qcaterpillar.readthedocs.io/en/latest/strategies.html).

Our LUT expansion uses tweedledum's positive-polarity Reed-Muller (PPRM) synthesis: write a small truth table as an XOR of products, then toggle the target once for each product. The same expansion is used for ordinary LUT mapping and best-fit mapping.

In [ ]:
method_table = pd.DataFrame([
    ["xag", "refactored XAG", "specialized XAG", "parity and controlled Rx/X"],
    ["aig_bennett", "original AIG", "Bennett", "AND/inverter gates"],
    ["xag_bennett", "refactored XAG", "Bennett", "AND/XOR gates"],
    ["klut_bennett", "k-LUT mapped from XAG", "Bennett", "PPRM per LUT"],
    ["best_fit", "XAG with nested LUT mapping", "best-fit / eager", "PPRM per LUT"],
], columns=["method", "representation", "schedule", "gate generation"])
display(method_table)

### The functions we will follow

1. `xor_and`: $y=(a\oplus b)\land c$. Watch how XOR becomes CNOTs.
2. `and4`: $y=a\land b\land c\land d$. This exposes the conjunction structure used in the borrowing demonstration.
3. `shared`: $p=a\oplus b$, $q=p\land c$, $y=q\land d$, $z=q\oplus d$. The two outputs share an intermediate result.

These teaching fixtures are included in the repository and are not EPFL benchmarks. For EPFL inputs, the XAG flow performs one fixed mockturtle bidecomposition refactoring pass. The AIG baseline keeps the original AIG. Therefore that comparison includes preprocessing differences; XAG versus XAG+Bennett shares the same refactored network.

In [ ]:
teaching = {}
checks = []
for name in ("xor_and", "and4", "shared"):
    teaching[name] = {}
    for method in METHODS:
        result = synthesize(ROOT / "datasets" / "boolean" / "teaching" / f"{name}.v", method)
        validate(result)
        verify_small_quantum(result)
        teaching[name][method] = result
        checks.append({"function": name, **result.summary()})
display(pd.DataFrame(checks)[["function", "method", "qubits", "workspace", "gates",
                               "exact_ccx", "controlled_rx_pi", "mcx_3plus", "validation"]])

### Read the Boolean graphs

Inputs enter from the left; output nodes are labeled `y0`, `y1`, and so on. An edge labeled NOT is inverted. The AIG and XAG below compute the same `xor_and` function, but their nodes differ.

In [ ]:
for method in ("aig_bennett", "xag"):
    display(Markdown(f"**{method}**"))
    fig = network_figure(teaching["xor_and"][method])
    display(fig)
    plt.close(fig)

### Read the quantum circuits

A controlled X with two controls is an exact Toffoli. A controlled $R_x(\pi)$ is **not** the same gate: it includes a phase on the activated branch. The native XAG flow pairs positive and negative rotations during computation and cleanup. Our adapter preserves those angles and control polarities.

Below are the complete `and4` computations. Raw output and scratch wires start in zero. A large controlled X is shown intact, so a one-gate row does not mean a one-gate hardware implementation.

In [ ]:
for method in METHODS:
    result = teaching["and4"][method]
    display(Markdown(f"**{method}** — inputs {result.metadata['input_qubits']}; "
                     f"output {result.metadata['output_qubits']}"))
    fig = result.circuit.draw("mpl", fold=90)
    display(fig)
    plt.close(fig)

### Keep a LUT visible, then expand it

The high-level LUT gate stores a truth table. Its displayed definition is still executable. Expanding the block exposes the controlled gates needed to implement it; neither view is a hardware cost model.

In [ ]:
result = teaching["shared"]["klut_bennett"]
for expanded in (False, True):
    display(Markdown("**Expanded PPRM gates**" if expanded else "**LUT blocks**"))
    fig = circuit_window(result, count=100, expand_luts=expanded)
    display(fig)
    plt.close(fig)

## 4. A common reversible oracle interface

The raw synthesis routines are allowed to assume clean output wires. Some also reuse those wires while computing intermediates. To obtain the familiar interface

$$|x\rangle|y\rangle|0\rangle \longmapsto
|x\rangle|y\oplus f(x)\rangle|0\rangle,$$

we append separate target wires and use **compute → copy → uncompute**. Copy here means CNOT into a target register, not cloning an arbitrary quantum state. The inverse restores every computation wire. This wrapper adds cost; we report raw and wrapped resources separately.

For a phase oracle, compute the outputs, apply Z to one selected output, and uncompute. That produces $(-1)^{f_i(x)}$ without leaving a result register behind.

In [ ]:
result = teaching["xor_and"]["xag"]
display(pd.DataFrame([
    {"view": "raw", "qubits": result.circuit.num_qubits, "gates": result.circuit.size()},
    {"view": "XOR oracle", "qubits": result.oracle().num_qubits, "gates": result.oracle().size()},
    {"view": "phase oracle for y0", "qubits": result.phase_oracle().num_qubits,
     "gates": result.phase_oracle().size()},
]))
fig = result.oracle().draw("mpl", fold=90)
display(fig)
plt.close(fig)
print(verify_small_quantum(result))

## 5. Synthesize the public benchmarks

This cell runs five approaches, with k = 3, 4, 6 for ordinary LUT mapping. Each run starts from the same benchmark file and checks that the transformed network is equivalent.

For these gate families, we can track classical bits **and phase factors** across every input assignment. We also check the expanded Qiskit circuit against the native records. This avoids an exponentially large statevector in the number of scratch wires. Tiny examples additionally undergo direct quantum simulation.

Timings are individual local runs, including conversion and potentially a cold LUT cache. They are useful for workflow sizing, not a rigorous runtime ranking.

In [ ]:
runs = {}
rows = []
configurations = [(m, 4) for m in METHODS if m != "klut_bennett"]
configurations += [("klut_bennett", k) for k in (3, 4, 6)]
for name, path in benchmarks.items():
    for method, k in configurations:
        result = synthesize(path, method, k=k)
        validate(result)
        label = f"klut{k}_bennett" if method == "klut_bennett" else method
        runs[name, label] = result
        rows.append({"benchmark": name, "configuration": label, **result.summary(),
                     "oracle_qubits": result.oracle().num_qubits,
                     "oracle_gates": result.oracle().size()})
summary = pd.DataFrame(rows)
display(summary[["benchmark", "configuration", "qubits", "workspace", "gates", "depth",
                 "exact_ccx", "controlled_rx_pi", "mcx_3plus", "validation"]])

### Compare structure without conflating gate costs

The left panel counts raw circuit wires. The right counts logical operations before hardware decomposition. An MCX with six controls and a CNOT each contribute one to that right-hand count, so it cannot decide which circuit is cheaper to execute.

The table retains exact Toffolis, controlled rotations, and larger MCX gates separately. A later fault-tolerant comparison must choose decompositions, permitted ancilla types, and any measurement-assisted cleanup model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
palette = ["#427f64", "#a74646", "#4a79a8", "#9b74a5", "#b98a36", "#419b9b", "#777777"]
for ax, metric, title in zip(axes, ("qubits", "gates"),
                             ("Raw circuit qubits", "Logical operations (mixed gate sizes)")):
    summary.pivot(index="benchmark", columns="configuration", values=metric).plot.bar(
        ax=ax, color=palette, rot=0)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.legend(fontsize=7)
fig.tight_layout()
display(fig)
plt.close(fig)

### Inspect one output cone

An output cone is the logic needed for a particular output. It is a derived function, not the full benchmark. Input ordering is preserved; unused input wires are hidden in the circuit drawing.

We choose the smallest nonconstant `ctrl` output cone that still contains an AND node. This gives a readable real-benchmark example. The full synthesized circuits remain available in `runs`.

In [ ]:
cones = [synthesize(benchmarks["ctrl"], outputs=[i]) for i in range(26)]
eligible = [r for r in cones if any(n["kind"] == "and" for n in r.metadata["network"]["nodes"])]
cone = min(eligible, key=lambda r: r.circuit.size())
validate(cone)
print("Derived ctrl output index:", cone.metadata["parameters"]["outputs"][0])
fig = network_figure(cone)
display(fig)
plt.close(fig)
fig = circuit_window(cone, count=25)
display(fig)
plt.close(fig)

### Inspect best-fit scheduling

The trace below comes from the reference mapping strategy. `lut_inputs` is the fan-in of an emitted cell; `compute` and `uncompute` show its role in the schedule. On small functions the best-fit result may be just one LUT per output, so the nested mapping has little room to act. We retain the actual trace rather than assuming every benchmark benefits from remapping.

In [ ]:
best_fit = runs["int2float", "best_fit"]
display(pd.DataFrame(best_fit.metadata["steps"]).head(16))
fig = circuit_window(best_fit, count=8, expand_luts=False)
display(fig)
plt.close(fig)

## 6. Find conditional-state facts

Suppose a clean qubit stores $t=a\land b$. On the branch where $t=1$, both a and b must be 1. Their unconditional state can still be a superposition. No measurement is required: the fact describes a subspace of the coherent computation.

The [conditionally clean ancilla paper, Section 3](https://arxiv.org/html/2407.17966v2) uses such facts to borrow system qubits. Three obligations matter:

- The borrowed qubit has a known value under a stored condition.
- The operation consuming the temporary result is protected by that condition; the condition itself remains valid.
- The borrowed qubit, the protecting workspace, and all relative phases are restored.

Our analysis recognizes conjunction gates whose target is initially zero over **all reachable inputs**. It checks the implication exactly and ends the candidate interval at the next write to the candidate or its protector. These are candidate facts, not certified circuit transformations. Correlations created by general LUTs and more sophisticated schedules may be missed.

In [ ]:
analysis = workspace_analysis(teaching["and4"]["aig_bennett"])
display(pd.DataFrame(analysis["candidates"]))
print(analysis["scope"])

candidate_rows = []
for label in summary["configuration"].unique():
    result = runs["ctrl", label]
    native_facts = workspace_analysis(result)
    expanded_facts = workspace_analysis(result, expanded=True)
    candidate_rows.append({
        "configuration": label,
        "native-gate candidates": len(native_facts["candidates"]),
        "expanded-gate candidates": len(expanded_facts["candidates"]),
        "peak potentially nonzero scratch": native_facts["peak_nonzero_workspace"],
    })
display(pd.DataFrame(candidate_rows))

### Follow workspace through time

This plot counts scratch wires that can be nonzero for at least one input assignment after each native operation. It excludes declared input and output wires. It is a semantic occupancy measure, not the allocator's live-qubit count: zero-valued wires can still be reserved.

Candidate counts also depend on representation. An opaque LUT hides its internal conjunctions until expansion, and multiple candidates may describe the same physical qubit at different times.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for label in ("xag", "xag_bennett", "klut4_bennett", "best_fit"):
    trace = workspace_analysis(runs["ctrl", label])
    ax.plot(trace["occupancy"], label=label)
ax.set_xlabel("Native operation index (different schedules)")
ax.set_ylabel("Potentially nonzero scratch")
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## 7. Borrow an input and prove restoration

For $a\land b\land c\land d$, the dedicated construction uses two clean scratch qubits. The conditional construction uses one:

1. Compute `flag = a AND b` into the clean scratch qubit.
2. Flip b. Under `flag = 1`, b is now zero.
3. Temporarily compute `c AND d` into b.
4. Toggle the target controlled by **both flag and b**.
5. Undo step 3, undo the flip, then uncompute flag.

When flag is zero, the temporary value of b need not equal `c AND d`. That does not affect the target because consumption is controlled by flag. Reversing the temporary computation restores b on that branch too.

This is a small exact-unitary demonstration of the paper's mechanism. It does not modify the benchmark circuits or claim a general optimizer.

In [ ]:
dedicated, borrowed = borrowing_demo()
for circuit in (dedicated, borrowed):
    display(Markdown(f"**{circuit.name}** — q0..q3: a,b,c,d; q4: target; q5 onward: clean scratch"))
    fig = circuit.draw("mpl", fold=100)
    display(fig)
    plt.close(fig)
display(pd.DataFrame(verify_borrowing()).T)

The verification compares every basis column of the subspace where the clean scratch starts in zero, including both target values. Equality of those columns checks the entire coherent action on that subspace, not just measurement probabilities. A separate simulation entangles an input with an external reference and checks that the reference coherence survives.

The result saves one clean qubit and keeps five exact Toffolis, at the cost of two X gates and a different depth. Replacing these gates with relative-phase or measurement-assisted constructions would require another proof.

## 8. Save results and extend the experiment

The exports contain both raw and wrapped circuits in QPY format, plus native gates, graphs, method parameters, source hashes, dependency versions, and verification results in JSON. CSV records the comparison table. QPY preserves circuit definitions without imposing a hardware basis.

In [ ]:
output_dir = OUT
for (name, label), result in runs.items():
    export_result(result, output_dir, f"{name}_{label}")
summary.to_csv(output_dir / "summary.csv", index=False)
print("Saved", len(runs), "runs to", output_dir)

# Optional medium benchmark: not part of the default exhaustive test set.
RUN_ROUTER = False
if RUN_ROUTER:
    router_path = download_benchmarks(("router",), offline=OFFLINE)["router"]
    router_rows = []
    for method in METHODS:
        result = synthesize(router_path, method)
        validate(result, sampled=True)
        router_rows.append(result.summary())
        export_result(result, output_dir, f"router_{method}")
    display(pd.DataFrame(router_rows))

## What this experiment establishes

We can now inspect how representation, LUT size, and cleanup strategy change the structures available for workspace reasoning. The borrowing example establishes one valid transformation; the benchmark analysis identifies places to investigate next.

A useful next experiment is to select a benchmark conjunction whose candidate interval is long enough for a specific controlled computation. Specify that computation, preserve its protecting condition, prove restoration and phase correctness, then compare it against allocating clean scratch or recomputing values under the same qubit budget.

Do not infer a T-count advantage from these logical gate counts. Exact Toffolis, paired controlled rotations, larger controlled gates, and measurement-assisted logical ANDs have different implementation assumptions.

### References

- [EPFL Combinational Benchmark Suite](https://github.com/lsils/benchmarks): original benchmark files and size overview.
- [Mockturtle network implementations](https://mockturtle.readthedocs.io/en/latest/implementations.html): AIG, XAG, and k-LUT representations.
- [Caterpillar mapping strategies](https://qcaterpillar.readthedocs.io/en/latest/strategies.html): Bennett, best-fit, and related schedules.
- [Khattar and Gidney, *Rise of conditionally clean ancillae for efficient quantum circuit constructions*, v2](https://arxiv.org/html/2407.17966v2): the conditional-workspace mechanism.
- Repository `REVIEW.md`: research objectives and the need to verify restoration and phase behavior.